<cell_type>markdown</cell_type># Dask Out-of-Core S3 Validation (30 GB)

This notebook validates Dask out-of-core processing with a **30 GB dataset** stored in S3.

**Key demonstration**: Processing 60 million spans (30 GB expanded) with a cluster that has
only 2-8 GB of memory, proving true out-of-core streaming capabilities.

**Note**: This notebook is read-only from ConfigMap. To edit:
```python
import shutil
shutil.copy('/home/jovyan/sample-notebooks/Dask_S3_Validation.ipynb', '/home/jovyan/')
```

In [ ]:
# Imports
import os
import uuid
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import s3fs
import dask.dataframe as dd
from dask.distributed import Client

import holoviews as hv
hv.extension('bokeh')

# AWS credentials are provided via:
# - IAM role (when running in AWS with proper instance profile)
# - Environment variables (AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_SESSION_TOKEN)
# - ~/.aws/credentials file
# No hardcoded credentials - uses boto3's credential chain
print("Using AWS credential chain (IAM role / env vars / credentials file)")

## 1. Connect to Dask Cluster

In [ ]:
# Connect to remote scheduler or create local
scheduler_address = os.getenv('DASK_SCHEDULER_ADDRESS')
if scheduler_address:
    print(f"Connecting to: {scheduler_address}")
    client = Client(scheduler_address)
else:
    print("Creating local cluster")
    client = Client(n_workers=2)

print(f"Dashboard: {client.dashboard_link}")
client

## 2. Configure S3 Access

In [ ]:
# S3 configuration — local lab uses RustFS (admin/admin, bucket cyberphy);
# AWS deploys use IAM / env vars from JupyterHub singleuser.extraEnv.
import os

_endpoint = os.getenv("S3_ENDPOINT") or ""
_key = os.getenv("AWS_ACCESS_KEY_ID") or None
_secret = os.getenv("AWS_SECRET_ACCESS_KEY") or None
_token = os.getenv("AWS_SESSION_TOKEN") or None

s3_opts = {"anon": False, "key": _key, "secret": _secret, "token": _token}
if _endpoint:
    s3_opts["client_kwargs"] = {"endpoint_url": _endpoint}
    s3_opts["config_kwargs"] = {
        "s3": {"addressing_style": "path"},
        "signature_version": "s3v4",
    }

s3 = s3fs.S3FileSystem(**s3_opts)

# Data bucket (default cyberphy for local RustFS lab)
S3_BUCKET = (
    os.getenv("S3_BUCKET")
    or os.getenv("OTEL_DATA_PATH", "s3://cyberphy/").replace("s3://", "").rstrip("/")
)
# OTEL_DATA_PATH may be s3://bucket/ or s3://bucket/prefix/
if S3_BUCKET.startswith("s3://"):
    S3_BUCKET = S3_BUCKET.replace("s3://", "").rstrip("/")
bucket_name = S3_BUCKET.split("/")[0]
data_prefix = "/".join(S3_BUCKET.split("/")[1:]) or "otel"

print(f"Endpoint: {_endpoint or 'AWS default'}")
print(f"Bucket:   {bucket_name}")
print(f"Prefix:   {data_prefix}")
print(f"Access:   {_key or '(IAM/default chain)'}")

# Test access
contents = s3.ls(bucket_name)
print(f"\nBucket contents: {contents}")


<cell_type>markdown</cell_type>## 3. Generate Large Synthetic Dataset (Idempotent)

We generate a **30 GB dataset** (expanded memory) to validate true out-of-core processing.
Each span includes additional attributes (~500 bytes/row in memory).

**Dataset Configuration:**
- Target: 30 GB expanded memory footprint
- Spans: ~60 million
- Partitions: 600 (100K spans each)
- Disk: ~6-8 GB (Parquet compressed with Snappy)

This dataset is typically **10-30× larger than cluster memory**, guaranteeing
that Dask must use streaming/out-of-core processing.

In [ ]:
# Fixed 30 GB dataset configuration - HARDCODED VALUES
BYTES_PER_SPAN = 500
TARGET_MEMORY_GB = 30
TARGET_MEMORY_BYTES = 30_000_000_000  # 30 GB

# HARDCODED: 60 million spans, 600 partitions
TOTAL_SPANS = 60_000_000
SPANS_PER_PARTITION = 100_000
NUM_PARTITIONS = 600
NUM_SERVICES = 20

# Get cluster info for comparison
scheduler_info = client.scheduler_info()
total_worker_memory = sum(
    w['memory_limit'] for w in scheduler_info['workers'].values()
)
num_workers = len(scheduler_info['workers'])
memory_ratio = TARGET_MEMORY_BYTES / total_worker_memory if total_worker_memory > 0 else float('inf')

print("="*60)
print("DATASET CONFIGURATION (30 GB)")
print("="*60)
print(f"\nDataset:")
print(f"  Target memory footprint: {TARGET_MEMORY_GB} GB")
print(f"  Total spans:             {TOTAL_SPANS:,}")
print(f"  Partitions:              {NUM_PARTITIONS}")
print(f"  Spans per partition:     {SPANS_PER_PARTITION:,}")
print(f"  Est. disk size:          {TOTAL_SPANS * 200 / 1e9:.1f} GB (compressed)")
print(f"\nCluster:")
print(f"  Workers:                 {num_workers}")
print(f"  Total worker memory:     {total_worker_memory/1e9:.2f} GB")
print(f"  Dataset/Memory ratio:    {memory_ratio:.1f}×")
print(f"\n  ⚡ Dataset is {memory_ratio:.1f}× larger than cluster memory!")

validation_path = f"{bucket_name}/{data_prefix}/validation-30gb"

# Check if data already exists WITH CORRECT PARTITION COUNT (idempotent)
try:
    existing_files = s3.glob(f"{validation_path}/*.parquet")
    if existing_files and len(existing_files) >= NUM_PARTITIONS:
        total_size = sum(s3.info(f)['size'] for f in existing_files)
        print(f"\n✓ Data exists: {len(existing_files)} files ({total_size/1024/1024/1024:.2f} GB)")
        print(f"  Path: s3://{validation_path}/")
        print("  Skipping generation (idempotent)")
        GENERATE_DATA = False
    elif existing_files:
        print(f"\n⚠ Partial data exists: {len(existing_files)} files (need {NUM_PARTITIONS})")
        print(f"  Deleting incomplete dataset and regenerating...")
        s3.rm(validation_path, recursive=True)
        GENERATE_DATA = True
    else:
        print(f"\n⚠ No existing data at: s3://{validation_path}/")
        print("  Will generate new dataset")
        GENERATE_DATA = True
except Exception as e:
    print(f"\n⚠ No existing data: {e}")
    GENERATE_DATA = True

In [ ]:
def generate_spans(n_spans=100000, n_services=20):
    """Generate synthetic span data with rich attributes."""
    services = [f"service-{i:02d}" for i in range(n_services)]
    operations = [
        'GET /api/v1/users', 'POST /api/v1/orders', 'GET /health',
        'PUT /api/v1/users/{id}', 'DELETE /api/v1/orders/{id}',
        'db.query.select', 'db.query.insert', 'db.query.update',
        'cache.get', 'cache.set', 'cache.delete',
        'queue.publish', 'queue.consume',
        'grpc.client.call', 'grpc.server.handle',
    ]
    http_methods = ['GET', 'POST', 'PUT', 'DELETE', 'PATCH']
    status_codes_http = [200, 201, 204, 400, 401, 403, 404, 500, 502, 503]

    now = datetime.now(timezone.utc)
    base_ts = int(now.timestamp() * 1e9)

    # Generate random data
    n = n_spans
    service_idx = np.random.randint(0, n_services, n)

    return pd.DataFrame({
        # Core span fields
        'trace_id': [uuid.uuid4().hex for _ in range(n)],
        'span_id': [uuid.uuid4().hex[:16] for _ in range(n)],
        'parent_span_id': np.where(
            np.random.random(n) > 0.3,
            [uuid.uuid4().hex[:16] for _ in range(n)],
            ''
        ),
        'service_name': np.array(services)[service_idx],
        'operation_name': np.random.choice(operations, n),

        # Timing
        'start_time_unix_nano': base_ts - np.random.randint(0, 86400 * 1e9, n).astype(np.int64),
        'duration_ns': np.random.exponential(scale=50_000_000, size=n).astype(np.int64),

        # Status
        'status_code': np.random.choice(['OK', 'ERROR', 'UNSET'], n, p=[0.92, 0.05, 0.03]),
        'http_status_code': np.random.choice(status_codes_http, n, p=[0.5, 0.1, 0.05, 0.1, 0.05, 0.05, 0.05, 0.05, 0.025, 0.025]),
        'http_method': np.random.choice(http_methods, n, p=[0.5, 0.25, 0.1, 0.1, 0.05]),

        # Additional attributes for richer data
        'user_id': [f"user-{np.random.randint(1, 10000):05d}" for _ in range(n)],
        'request_id': [uuid.uuid4().hex for _ in range(n)],
        'host': np.array([f"{services[i]}-{np.random.randint(1,10)}.cluster.local" for i in service_idx]),
        'db_statement': np.where(
            np.random.random(n) > 0.7,
            [f"SELECT * FROM table_{np.random.randint(1,100)} WHERE id = {np.random.randint(1,1000000)}" for _ in range(n)],
            ''
        ),
        'error_message': np.where(
            np.random.random(n) > 0.95,
            np.random.choice(['Connection timeout', 'Rate limit exceeded', 'Internal error', 'Not found', 'Permission denied'], n),
            ''
        ),
    })

if GENERATE_DATA:
    import time as time_module
    
    print(f"Generating {TOTAL_SPANS:,} spans across {NUM_PARTITIONS} partitions...")
    print(f"  {SPANS_PER_PARTITION:,} spans per partition")
    print(f"  {NUM_SERVICES} services")
    print(f"  Estimated time: {NUM_PARTITIONS * 2 / 60:.0f}-{NUM_PARTITIONS * 4 / 60:.0f} minutes")
    print()
    
    start_time = time_module.time()
    bytes_written = 0

    for partition in range(NUM_PARTITIONS):
        df = generate_spans(n_spans=SPANS_PER_PARTITION, n_services=NUM_SERVICES)

        # Convert to PyArrow table
        table = pa.Table.from_pandas(df, preserve_index=False)

        # Write to S3
        output_path = f"s3://{validation_path}/partition_{partition:04d}.parquet"
        pq.write_table(table, output_path, filesystem=s3, compression='snappy')
        
        # Track progress
        file_size = s3.info(f"{validation_path}/partition_{partition:04d}.parquet")['size']
        bytes_written += file_size
        
        # Progress update every 50 partitions
        if (partition + 1) % 50 == 0:
            elapsed = time_module.time() - start_time
            rate = (partition + 1) / elapsed
            eta = (NUM_PARTITIONS - partition - 1) / rate
            print(f"  [{partition + 1:4d}/{NUM_PARTITIONS}] "
                  f"{bytes_written/1e9:.2f} GB written, "
                  f"{rate:.1f} partitions/sec, "
                  f"ETA: {eta/60:.1f} min")

    elapsed = time_module.time() - start_time
    
    # Print final stats
    files = s3.glob(f"{validation_path}/*.parquet")
    total_size = sum(s3.info(f)['size'] for f in files)
    
    print(f"\n{'='*60}")
    print("DATA GENERATION COMPLETE")
    print(f"{'='*60}")
    print(f"  Files:       {len(files)}")
    print(f"  Total size:  {total_size/1024/1024/1024:.2f} GB (compressed)")
    print(f"  Spans:       {TOTAL_SPANS:,}")
    print(f"  Time:        {elapsed/60:.1f} minutes")
    print(f"  Throughput:  {TOTAL_SPANS/elapsed:,.0f} spans/sec")
else:
    print("Using existing data")

## 4. Load Data with Dask (Lazy)

In [ ]:
# Load as Dask DataFrame (lazy - no data loaded yet)
ddf = dd.read_parquet(
    f"s3://{validation_path}/*.parquet",
    storage_options=s3_opts,  # Use IAM role
)

print(f"Partitions: {ddf.npartitions}")
print(f"Columns: {list(ddf.columns)}")
print(f"Expected rows: {TOTAL_SPANS:,}")
print(f"\n⚡ This is LAZY - no data loaded yet!")
print("   Data will only be fetched when .compute() is called")
ddf

## 4b. PyArrow Dataset API (Zero-Copy, Predicate Pushdown)

PyArrow's Dataset API provides fine-grained control over Parquet reading with:
- **Schema inspection** without reading data
- **Predicate pushdown** - filters applied at the storage layer (only matching row groups read)
- **Column projection** - only requested columns are deserialized
- **Zero-copy reads** - memory-mapped access when possible

This is the foundation that Dask uses internally, but explicit access enables optimization.

In [ ]:
import pyarrow.dataset as ds
import pyarrow.fs as pafs

# Create PyArrow S3 filesystem (uses same credential chain as s3fs)
arrow_kwargs = dict(region=os.getenv('AWS_REGION', os.getenv('S3_REGION', 'us-east-1')))
if _key and _secret:
    arrow_kwargs['access_key'] = _key
    arrow_kwargs['secret_key'] = _secret
if _token:
    arrow_kwargs['session_token'] = _token
if _endpoint:
    arrow_kwargs['endpoint_override'] = _endpoint
    arrow_kwargs['scheme'] = 'https' if _endpoint.startswith('https') else 'http'
arrow_fs = pafs.S3FileSystem(**arrow_kwargs)

# Load as Arrow Dataset - NO DATA READ YET (just metadata)
dataset = ds.dataset(
    f"{validation_path}/",
    format="parquet",
    filesystem=arrow_fs,
)

print("="*60)
print("ARROW DATASET METADATA (no data loaded)")
print("="*60)
print(f"\nFiles: {len(dataset.files)}")

# Schema inspection - critical for understanding data without loading
print(f"\nSchema ({len(dataset.schema)} columns):")
for field in dataset.schema:
    print(f"  {field.name:30} {field.type}")

# Get row group and size info from fragments
total_row_groups = 0
for fragment in dataset.get_fragments():
    total_row_groups += fragment.metadata.num_row_groups
    
print(f"\nTotal row groups: {total_row_groups}")
print(f"Parquet files: {len(dataset.files)}")

In [ ]:
%%time
# PREDICATE PUSHDOWN - filter applied at Parquet row-group level
# Only matching row groups are read from S3 (massive I/O savings)

# Filter: only ERROR spans with high latency (> 100ms)
error_filter = (
    (ds.field('status_code') == 'ERROR') & 
    (ds.field('duration_ns') > 100_000_000)  # > 100ms
)

# Column projection - only read columns we need (reduces I/O and memory)
selected_columns = ['trace_id', 'service_name', 'operation_name', 'duration_ns', 'error_message']

# Execute with pushdown - this reads ONLY matching data from S3
error_table = dataset.to_table(
    filter=error_filter,
    columns=selected_columns,
)

print("="*60)
print("PREDICATE PUSHDOWN + COLUMN PROJECTION")
print("="*60)
print(f"\nFilter: status_code='ERROR' AND duration_ns > 100ms")
print(f"Columns: {selected_columns}")
print(f"\nResult: {len(error_table):,} rows (from {TOTAL_SPANS:,} total)")
print(f"Reduction: {(1 - len(error_table)/TOTAL_SPANS)*100:.1f}% of rows filtered at storage layer")
print(f"Memory: {error_table.nbytes/1024/1024:.1f} MB (vs {TARGET_MEMORY_GB} GB for full dataset)")

# Show sample
error_df = error_table.to_pandas()
print(f"\nSample high-latency errors:")
error_df['duration_ms'] = error_df['duration_ns'] / 1e6
error_df[['service_name', 'operation_name', 'duration_ms', 'error_message']].head(10)

In [ ]:
%%time
# ARROW + DASK: Distributed processing with predicate pushdown
# Dask can use Arrow's predicate pushdown when reading Parquet

# Read with filters pushed down to Arrow (distributed across workers)
# Each worker reads only matching row groups from S3
ddf_filtered = dd.read_parquet(
    f"s3://{validation_path}/*.parquet",
    storage_options=s3_opts,
    filters=[
        ('status_code', '==', 'ERROR'),
    ],
    columns=['trace_id', 'service_name', 'operation_name', 'duration_ns', 'error_message', 'status_code'],
    engine='pyarrow',  # Explicit Arrow engine
)

print("="*60)
print("DASK + ARROW PREDICATE PUSHDOWN (Distributed)")
print("="*60)
print(f"\nPartitions: {ddf_filtered.npartitions}")
print(f"Filter: status_code='ERROR' (pushed to Arrow)")
print(f"This is LAZY - filter will be applied during compute()")

# Compute aggregation on filtered data (distributed)
error_by_service = ddf_filtered.groupby('service_name').agg({
    'trace_id': 'count',
    'duration_ns': ['mean', 'max'],
}).compute()

error_by_service.columns = ['error_count', 'mean_duration_ns', 'max_duration_ns']
error_by_service['mean_duration_ms'] = error_by_service['mean_duration_ns'] / 1e6
error_by_service = error_by_service.sort_values('error_count', ascending=False)

print(f"\n✓ Distributed aggregation on filtered data:")
print(f"  Total errors: {error_by_service['error_count'].sum():,}")
error_by_service[['error_count', 'mean_duration_ms']]

In [ ]:
# SCANNER API: Fine-grained batch iteration for streaming processing
# This is the most memory-efficient way to process large datasets

print("="*60)
print("ARROW SCANNER: Streaming Batch Processing")
print("="*60)

# Create scanner with batch size control
scanner = dataset.scanner(
    columns=['service_name', 'duration_ns', 'status_code'],
    filter=(ds.field('http_status_code') >= 500),  # Server errors only
    batch_size=100_000,  # 100K rows per batch
)

# Process in batches - never loads full dataset
batch_count = 0
total_rows = 0
duration_sum = 0

for batch in scanner.to_batches():
    batch_count += 1
    total_rows += len(batch)
    # Aggregate within batch (zero-copy access to Arrow arrays)
    duration_sum += batch.column('duration_ns').to_numpy().sum()

avg_duration_ms = (duration_sum / total_rows / 1e6) if total_rows > 0 else 0

print(f"\nProcessed {batch_count} batches ({total_rows:,} rows)")
print(f"Average latency for HTTP 5xx errors: {avg_duration_ms:.2f} ms")
print(f"\n✓ Memory-efficient: only one batch in memory at a time")
print(f"  Each batch: ~{100_000 * 50 / 1024 / 1024:.1f} MB")
print(f"  Full dataset: {TARGET_MEMORY_GB} GB")
print(f"  Memory savings: {TARGET_MEMORY_GB * 1000 / (100_000 * 50 / 1024 / 1024):.0f}×")

## 5. Compute Aggregations (Distributed)

In [ ]:
%%time
# This triggers distributed computation across all workers
# Watch the Dask dashboard to see tasks being distributed!
total_spans = len(ddf)
print(f"✓ Total spans: {total_spans:,}")
print(f"  Processing {total_spans:,} rows across {ddf.npartitions} partitions")

In [ ]:
%%time
# Multi-dimensional aggregation - watch the Dask dashboard!
# This demonstrates streaming aggregation: data flows through workers
# without loading the entire dataset into memory

service_stats = ddf.groupby('service_name').agg({
    'span_id': 'count',
    'duration_ns': ['mean', 'max', 'min', 'std'],
    'http_status_code': ['mean'],
}).compute()

service_stats.columns = ['count', 'mean_duration_ns', 'max_duration_ns', 'min_duration_ns', 'std_duration_ns', 'avg_http_status']
service_stats['mean_duration_ms'] = service_stats['mean_duration_ns'] / 1e6
service_stats['max_duration_ms'] = service_stats['max_duration_ns'] / 1e6
service_stats = service_stats.sort_values('count', ascending=False)

print(f"✓ Aggregated {total_spans:,} spans into {len(service_stats)} service groups")
service_stats[['count', 'mean_duration_ms', 'max_duration_ms', 'std_duration_ns']]

In [ ]:
%%time
# Error rate by service - OPTIMIZED (no apply/lambda)
# Count errors and total per service, then compute ratio
import dask

error_counts = ddf[ddf['status_code'] == 'ERROR'].groupby('service_name').size()
total_counts = ddf.groupby('service_name').size()

# Compute both in parallel
error_counts, total_counts = dask.compute(error_counts, total_counts)

# Calculate error rate
error_rate = (error_counts / total_counts * 100).fillna(0).sort_values(ascending=False)

print("Error rate by service (%)")
error_rate

## 6. Visualizations with HoloViews

In [ ]:
# Sample for visualization (compute a subset)
sample_df = ddf.sample(frac=0.1).compute()
print(f"Sample size: {len(sample_df):,} spans")

In [ ]:
# Duration distribution by service
sample_df['duration_ms'] = sample_df['duration_ns'] / 1e6

hv.BoxWhisker(sample_df, kdims='service_name', vdims='duration_ms').opts(
    width=800, height=400,
    title='Latency Distribution by Service',
    ylabel='Duration (ms)',
    box_fill_color=hv.dim('service_name').categorize(dict(zip(
        sample_df['service_name'].unique(),
        hv.Cycle('Category10').values
    )))
)

In [ ]:
# Span count by service (bar chart)
counts = sample_df.groupby('service_name').size().reset_index(name='count')

hv.Bars(counts, kdims='service_name', vdims='count').opts(
    width=600, height=400,
    title='Span Count by Service',
    color='service_name',
    cmap='Category10',
    xrotation=45,
)

In [ ]:
# Status distribution
status_counts = sample_df.groupby(['service_name', 'status_code']).size().reset_index(name='count')

hv.Bars(status_counts, kdims=['service_name', 'status_code'], vdims='count').opts(
    width=800, height=400,
    title='Status Code Distribution by Service',
    xrotation=45,
    color='status_code',
    cmap={'OK': 'green', 'ERROR': 'red', 'UNSET': 'gray'},
)

## 6b. Hierarchical Drill-Down Visualization

This section demonstrates **zoom-triggered aggregation** - a key pattern for exploring
large temporal + high-dimensional datasets like OTel traces.

**How it works:**
- **Zoomed out** (hours/days): Data aggregated at region/AZ level
- **Medium zoom** (minutes): Service-level aggregates
- **Zoomed in** (seconds): Individual spans visible

This enables exploring data from cluster-wide overview down to individual requests
without loading the entire dataset into memory.

In [ ]:
# Import drill-down visualization utilities
from holoviews.operation.datashader import rasterize, datashade
from holoviews.streams import RangeXY, RangeX

# Prepare data with proper datetime column
drill_df = sample_df.copy()
drill_df['timestamp'] = pd.to_datetime(drill_df['start_time_unix_nano'], unit='ns')
drill_df['duration_ms'] = drill_df['duration_ns'] / 1e6

print(f"Prepared {len(drill_df):,} spans for drill-down visualization")
print(f"Time range: {drill_df['timestamp'].min()} to {drill_df['timestamp'].max()}")

In [ ]:
# Create Service Latency Heatmap with linked histogram
# Zoom/pan on the heatmap to filter the histogram

def create_latency_heatmap(df):
    """Create an interactive latency heatmap by service over time."""
    # Bucket by 1-minute intervals
    df = df.copy()
    df['time_bucket'] = df['timestamp'].dt.floor('1min')
    
    # Aggregate: mean latency per service per time bucket
    agg = df.groupby(['service_name', 'time_bucket']).agg({
        'duration_ms': ['mean', 'count', 'max'],
    }).reset_index()
    agg.columns = ['service', 'time', 'mean_latency', 'count', 'max_latency']
    
    # Create heatmap
    heatmap = hv.HeatMap(agg, kdims=['time', 'service'], vdims=['mean_latency', 'count'])
    
    return heatmap.opts(
        title='Service Latency Heatmap (zoom to filter histogram →)',
        colorbar=True,
        cmap='RdYlGn_r',  # Red=slow, Green=fast
        tools=['hover', 'box_select'],
        width=700,
        height=400,
        xrotation=45,
    )

# Create the linked view
heatmap = create_latency_heatmap(drill_df)

# Dynamic histogram that updates based on heatmap selection
range_stream = RangeXY(source=heatmap)

def update_histogram(x_range, y_range):
    """Update histogram based on selected time range."""
    filtered = drill_df
    
    if x_range and x_range[0] is not None:
        mask = (drill_df['timestamp'] >= pd.Timestamp(x_range[0])) & \
               (drill_df['timestamp'] <= pd.Timestamp(x_range[1]))
        filtered = drill_df[mask]
    
    if filtered.empty:
        return hv.Histogram([]).opts(title='Latency Distribution (No data)')
    
    frequencies, edges = np.histogram(filtered['duration_ms'], bins=50)
    hist = hv.Histogram((edges, frequencies))
    
    return hist.opts(
        title=f'Latency Distribution (n={len(filtered):,})',
        xlabel='Latency (ms)',
        ylabel='Count',
        width=350,
        height=350,
        fill_color='#3182bd',
    )

histogram = hv.DynamicMap(update_histogram, streams=[range_stream])

# Display linked views side by side
(heatmap + histogram).opts(shared_axes=False)

In [ ]:
# Overview + Detail View with Hierarchy-Aware Aggregation
# The detail view changes aggregation level based on zoom

def determine_agg_level(time_range_seconds):
    """Determine aggregation level from visible time range."""
    if time_range_seconds > 3600:    # > 1 hour
        return 'service_group', '10min'   # Group services, 10-min buckets
    elif time_range_seconds > 300:   # > 5 minutes
        return 'service', '1min'          # Individual services, 1-min buckets
    else:                            # < 5 minutes
        return 'host', '10s'              # Host level, 10-sec buckets

def create_overview(df):
    """Create overview area chart showing total request volume."""
    hourly = df.groupby(df['timestamp'].dt.floor('10min')).agg({
        'duration_ms': ['mean', 'count']
    }).reset_index()
    hourly.columns = ['time', 'mean_latency', 'count']
    
    area = hv.Area(hourly, kdims=['time'], vdims=['count'])
    return area.opts(
        height=120, width=900,
        alpha=0.7, color='#1f77b4',
        title='Overview: Request Volume (drag to zoom detail view)',
        tools=['xbox_select'],
    )

def create_detail(df, x_range):
    """Create detail view with hierarchy-aware aggregation."""
    if x_range is None or x_range[0] is None:
        # Default view
        time_range_sec = 3600
        filtered = df
    else:
        start, end = pd.Timestamp(x_range[0]), pd.Timestamp(x_range[1])
        time_range_sec = (end - start).total_seconds()
        filtered = df[(df['timestamp'] >= start) & (df['timestamp'] <= end)]
    
    if filtered.empty:
        return hv.Scatter([]).opts(title='Detail (No data in range)')
    
    # Determine aggregation level
    group_col, time_bucket = determine_agg_level(time_range_sec)
    
    if group_col == 'service_group':
        # Group services into clusters (e.g., service-0X -> group-0)
        filtered = filtered.copy()
        filtered['_group'] = filtered['service_name'].str.extract(r'service-(\d)')[0].apply(
            lambda x: f'group-{x}' if pd.notna(x) else 'other'
        )
        group_by = '_group'
    elif group_col == 'host':
        group_by = 'host'
    else:
        group_by = 'service_name'
    
    # Aggregate
    filtered = filtered.copy()
    filtered['_bucket'] = filtered['timestamp'].dt.floor(time_bucket)
    
    agg = filtered.groupby([group_by, '_bucket']).agg({
        'duration_ms': 'mean',
    }).reset_index()
    agg.columns = ['group', 'time', 'latency']
    
    # Create scatter plot
    scatter = hv.Scatter(agg, kdims=['time'], vdims=['latency', 'group'])
    
    return scatter.opts(
        title=f'Detail: {group_col} level, {time_bucket} buckets (n={len(agg):,})',
        color='group',
        cmap='Category20',
        width=900, height=350,
        tools=['hover'],
        size=8,
        alpha=0.7,
    )

# Create the linked overview + detail
overview = create_overview(drill_df)
range_x = RangeX(source=overview)

detail = hv.DynamicMap(
    lambda x_range: create_detail(drill_df, x_range),
    streams=[range_x]
)

# Stack vertically
(overview + detail).cols(1).opts(shared_axes=False)

In [ ]:
# Datashader visualization for large datasets
# This uses server-side rendering to handle millions of points efficiently

from holoviews.operation.datashader import datashade, dynspread

# Load a sample for datashader demo (10% = ~6M points from 60M)
print("Loading ~6M spans for datashader visualization...")
large_sample = ddf.sample(frac=0.1).compute()
large_sample['timestamp'] = pd.to_datetime(large_sample['start_time_unix_nano'], unit='ns')
large_sample['duration_ms'] = large_sample['duration_ns'] / 1e6
large_sample['timestamp_numeric'] = large_sample['timestamp'].astype(np.int64) / 1e9  # seconds
print(f"Loaded {len(large_sample):,} spans")

# Create points - each span is a point in (time, latency) space
points = hv.Points(
    large_sample,
    kdims=['timestamp_numeric', 'duration_ms'],
    vdims=['service_name']
)

# Datashade - aggregates millions of points to screen resolution
# Automatically re-aggregates on zoom/pan
shaded = datashade(points, cmap='fire').opts(
    title=f'Datashaded Latency: {len(large_sample):,} spans (zoom to explore)',
    width=900,
    height=400,
    tools=['hover', 'wheel_zoom', 'box_zoom', 'reset'],
    xlabel='Time (Unix seconds)',
    ylabel='Latency (ms)',
)

# dynspread makes individual points visible when zoomed in
dynspread(shaded, threshold=0.5, max_px=5)

<cell_type>markdown</cell_type>## 7. Out-of-Core Stress Test (30 GB)

This section demonstrates that Dask processes data **much larger than worker memory** using streaming aggregation.
With a 30 GB dataset and typical 2-8 GB cluster memory, this is a 4-15× memory challenge.

Each worker processes partitions sequentially, computing partial results and releasing memory.

In [ ]:
import time

def show_worker_memory(label=""):
    """Display worker memory utilization."""
    info = client.scheduler_info()
    print(f"\n{'='*60}")
    print(f"Worker Memory Utilization {label}")
    print(f"{'='*60}")
    
    total_used = 0
    total_limit = 0
    
    for worker_id, worker in sorted(info['workers'].items()):
        mem_used = worker.get('metrics', {}).get('memory', 0)
        mem_limit = worker['memory_limit']
        pct = (mem_used / mem_limit * 100) if mem_limit else 0
        worker_short = worker_id.split('/')[-1][:25]
        bar = '█' * int(pct/5) + '░' * (20 - int(pct/5))
        print(f"  {worker_short:25} [{bar}] {mem_used/1e9:.2f}GB / {mem_limit/1e9:.2f}GB ({pct:5.1f}%)")
        total_used += mem_used
        total_limit += mem_limit
    
    print(f"\n  TOTAL: {total_used/1e9:.2f}GB / {total_limit/1e9:.2f}GB ({total_used/total_limit*100:.1f}%)")
    return total_used, total_limit

# Show baseline memory and dataset ratio
baseline_used, baseline_limit = show_worker_memory("(baseline)")

print(f"\n{'='*60}")
print("OUT-OF-CORE CHALLENGE: 30 GB DATASET")
print(f"{'='*60}")
print(f"\n  Dataset expanded size:   {TARGET_MEMORY_GB} GB")
print(f"  Total worker memory:     {baseline_limit/1e9:.2f} GB")
print(f"  Ratio (dataset/workers): {TARGET_MEMORY_BYTES/baseline_limit:.1f}×")
print(f"\n  ⚡ Dataset is {TARGET_MEMORY_BYTES/baseline_limit:.1f}× larger than cluster memory!")
print(f"     Dask MUST use out-of-core streaming to process this.")

In [ ]:
%%time
# Complex multi-key aggregation - stresses out-of-core processing
# This groups by multiple columns, requiring more intermediate state

print(f"Running complex aggregation across {TOTAL_SPANS:,} spans (30 GB)...")
print("Watch the Dask dashboard - you'll see tasks streaming through workers\n")

# First, add a column for HTTP errors (status >= 400)
# This avoids using lambda in agg() which Dask can't serialize
ddf_with_errors = ddf.assign(is_http_error=(ddf['http_status_code'] >= 400).astype(int))

complex_stats = ddf_with_errors.groupby(['service_name', 'operation_name', 'http_method']).agg({
    'span_id': 'count',
    'duration_ns': ['mean', 'max', 'std'],
    'is_http_error': 'sum',  # Sum of boolean = count of errors
}).compute()

complex_stats.columns = ['count', 'mean_duration_ns', 'max_duration_ns', 'std_duration_ns', 'error_count']
complex_stats['mean_duration_ms'] = complex_stats['mean_duration_ns'] / 1e6
complex_stats['error_rate_pct'] = complex_stats['error_count'] / complex_stats['count'] * 100
complex_stats = complex_stats.sort_values('count', ascending=False)

print(f"✓ Created {len(complex_stats)} distinct groups from {total_spans:,} spans")
print(f"  Unique service/operation/method combinations: {len(complex_stats)}")
complex_stats.head(15)[['count', 'mean_duration_ms', 'error_rate_pct']]

In [ ]:
# Check memory AFTER complex aggregation
# Key insight: memory should still be bounded even after processing 30 GB dataset
post_used, post_limit = show_worker_memory("(after complex aggregation)")

print("\n" + "="*60)
print("OUT-OF-CORE VERIFICATION: 30 GB DATASET")
print("="*60)

# Calculate memory headroom
max_worker_pct = max(
    w.get('metrics', {}).get('memory', 0) / w['memory_limit'] * 100
    for w in client.scheduler_info()['workers'].values()
)

print(f"""
Dataset size:          {TARGET_MEMORY_GB} GB ({TOTAL_SPANS:,} spans)
Cluster memory:        {post_limit/1e9:.1f} GB
Dataset/Memory ratio:  {TARGET_MEMORY_BYTES/post_limit:.1f}×
Peak worker memory:    {max_worker_pct:.1f}%
Memory stayed bounded: {'✓ YES' if max_worker_pct < 90 else '✗ NO'}

How out-of-core processing works:
  1. Dask reads partitions lazily from S3 (one at a time per task)
  2. Each partition ({SPANS_PER_PARTITION:,} spans, ~50 MB) is processed
  3. Partial aggregation results accumulated, raw data released
  4. Only final aggregated results stay in memory

Result: Processed {TARGET_MEMORY_GB} GB dataset using {post_limit/1e9:.1f} GB cluster!
        That's {TARGET_MEMORY_BYTES/post_limit:.0f}× more data than available RAM.
""")

## 8. Cleanup

In [ ]:
# Optionally delete test data
# Uncomment to clean up:
# s3.rm(validation_path, recursive=True)
# print(f"Deleted: {validation_path}")

In [ ]:
# Close client
# client.close()

<cell_type>markdown</cell_type>---

## Validation Summary

This notebook validated **Dask out-of-core processing** with a **30 GB dataset**:

| Metric | Value |
|--------|-------|
| Dataset Size | **30 GB** (expanded memory) |
| Total Spans | 60,000,000 |
| Partitions | 600 |
| Disk Size | ~6-8 GB (Parquet/Snappy) |
| Services | 20 |

### Key Validation: 30 GB Out-of-Core Processing

The 30 GB dataset is typically **10-30× larger** than cluster memory:
- With 2 workers × 2 GB = 4 GB cluster → **7.5× ratio**
- With 4 workers × 2 GB = 8 GB cluster → **3.75× ratio**
- With 2 workers × 1 GB = 2 GB cluster → **15× ratio**

This **guarantees** Dask must use streaming/out-of-core processing - the data
simply cannot fit in memory.

### Capabilities Demonstrated

1. **Dask Cluster Connection** - Connected to distributed scheduler with multiple workers
2. **S3 Access via IAM** - Read/write to S3 using AWS IAM role credentials
3. **30 GB Dataset Generation** - 60M spans across 600 partitions
4. **Idempotent Data Generation** - Skips regeneration if data already exists
5. **Lazy Loading** - 60M rows loaded only when `.compute()` is called
6. **Distributed Aggregations** - GroupBy operations computed in parallel across workers
7. **Out-of-Core Processing** - Memory stays bounded processing 30 GB with ~2-8 GB cluster
8. **HoloViews Visualizations** - Interactive plots render correctly in JupyterLab
9. **Hierarchical Drill-Down** - Zoom-triggered aggregation (service group → service → host)
10. **Datashader Integration** - Millions of points rendered efficiently with zoom/pan

### PyArrow Dataset Capabilities

| Feature | Description | Benefit |
|---------|-------------|---------|
| **Schema Inspection** | Read schema without loading data | Fast exploration of unknown datasets |
| **Predicate Pushdown** | Filters applied at Parquet row-group level | Only matching data read from S3 |
| **Column Projection** | Only requested columns deserialized | Reduced I/O and memory |
| **Scanner API** | Batch-by-batch iteration | Constant memory for any dataset size |
| **Dask Integration** | Filters passed to Arrow engine | Distributed + pushdown combined |

### Data Loading Approaches Compared

| Approach | Use Case | Memory Model |
|----------|----------|--------------|
| `dd.read_parquet()` | Distributed aggregations | Partitioned across workers |
| `ds.dataset().to_table()` | Single-node filtered reads | Predicate pushdown, full result in memory |
| `ds.dataset().scanner()` | Streaming processing | Constant memory (batch-at-a-time) |
| `dd.read_parquet(filters=)` | Distributed + filtered | Best of both worlds |

### Drill-Down Visualization Pattern

The notebook demonstrates **zoom-triggered hierarchy** for exploring large OTel datasets:

| Zoom Level | Aggregation | Time Bucket | Use Case |
|------------|-------------|-------------|----------|
| Zoomed out (>1h) | Service groups | 10 min | Cluster-wide overview |
| Medium (5m-1h) | Individual services | 1 min | Service comparison |
| Zoomed in (<5m) | Hosts/pods | 10 sec | Incident investigation |

This pattern enables exploring from region/AZ-level down to individual spans without
loading the entire dataset into memory.

### Key Insight

**No bottlenecks** in the data path:

1. **S3 → Arrow**: Predicate pushdown means only relevant row groups are read
2. **Arrow → Dask**: Zero-copy handoff, data stays in Arrow format
3. **Dask Workers**: Stream partitions, compute partial aggregations, release memory
4. **Final Result**: Only aggregated summaries returned to client

This architecture processes datasets **orders of magnitude larger than cluster memory**.